# 01. Khảo sát Dữ liệu & Kiểm toán Data Leakage trước Huấn luyện (Tuần 5)
**Dự án:** Smart LMS & Generative AI – Phân tích dữ liệu người học và tạo nội dung học tập tự động  
**Môn học trọng tâm:** `FFF` – Học kỳ `2013J` (2.283 sinh viên)


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../work/prj1/prj1/week 3+4/data/output/student_features.csv')
print("Shape:", df.shape)
print("Duplicates:", df.duplicated().sum())
print("Target final_result:\n", df['final_result'].value_counts())
print("Target is_at_risk:\n", df['is_at_risk'].value_counts(normalize=True)*100)


Shape: (2283, 37)
Duplicates: 0
Target final_result:
 Pass           908
Withdrawn      675
Fail           513
Distinction    187
Name: count, dtype: int64
Target is_at_risk:
 1    52.036794
0    47.963206
Name: proportion, dtype: float64


## Kiểm toán Data Leakage (Mục 1.4 & 2.1)
1. **Phát hiện 1 (Cấu trúc 37 cột vs 24 đặc trưng):** File `student_features.csv` chứa 37 cột, bao gồm 3 cột định danh (`code_module`, `code_presentation`, `id_student`), 2 cột mục tiêu (`final_result`, `is_at_risk`), 2 cột rò rỉ trực tiếp nhãn rút môn (`date_unregistration`, `has_unregistered`: 674/675 sinh viên Withdrawn có `has_unregistered=1`), 6 cột nhân khẩu học phân loại (`imd_band` còn 101 giá trị `NaN`), và **24 đặc trưng định lượng đầu vào** (`0` giá trị thiếu).
2. **Phát hiện 2 (Rò rỉ thời gian sau Ngày 100):** Trong `feature_engineering.py` (Tuần 4), dù đã loại bỏ `Exam` (ngày 236), các đặc trưng điểm số (`assessment_completion_rate`, `weighted_score`) và tương tác VLE tổng vẫn tính cả TMA 4 (ngày 131), TMA 5 (ngày 173) và các CMA nộp sau ngày 100. Do đó, dự án xây dựng thêm bộ dữ liệu **Cảnh báo sớm nghiêm ngặt tại Ngày 100 (`student_features_early_d100.csv`)** chỉ sử dụng dữ liệu `date <= 100` để đối chứng công bằng.


In [1]:
print("Crosstab has_unregistered vs final_result:")
print(pd.crosstab(df['has_unregistered'], df['final_result'], margins=True))


Crosstab has_unregistered vs final_result:
final_result      Distinction  Fail  Pass  Withdrawn   All
has_unregistered                                          
0                         187   510   908          1  1606
1                           0     3     0        674   677
All                       187   513   908        675  2283
